# Case study: past-only aggregates and leaking features in the Berlin Airbnb data

*Session 9, block 2 practice. Author: course team, licence CC-BY-4.0.*

Two questions, two kinds of leakage:

1. **Which listings will be busy next year?** Features from the joined table `reviews_monthly`, computed as of a cut-off date, against snapshot columns that already contain the period to be predicted.
2. **What is a listing worth per night?** A revenue column that Inside Airbnb derives from the price.

Theory: [page 2](../theory/02-aggregates-leakage-and-text-statistics.md).

**Data.** Inside Airbnb, Berlin, snapshot of 26 June 2026 (CC BY 4.0), prepared with `uv run python case-study/prepare_airbnb.py`; no host names; report in aggregate. Reviews are a proxy for stays. The notebook runs in about a minute.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
AIRBNB = ROOT / "case-study" / "data" / "airbnb"
import json

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, cross_val_score

listings = pd.read_parquet(AIRBNB / "listings.parquet")
reviews = pd.read_parquet(AIRBNB / "reviews_monthly.parquet")
print(listings.shape, reviews.shape, reviews["month"].min().date(), reviews["month"].max().date())
gbm = HistGradientBoostingRegressor(random_state=0)
cv = GroupKFold(5)

## 1. Demand features as of a cut-off date

A function that builds the features and the target for any cut-off: features from the review months **before** the cut-off, target = reviews in the 12 months **from** the cut-off.

In [ ]:
def demand_table(cutoff):
    cutoff = pd.Timestamp(cutoff)
    past = reviews[reviews["month"] < cutoff]
    future = reviews[(reviews["month"] >= cutoff) & (reviews["month"] < cutoff + pd.DateOffset(months=12))]
    g = past.groupby("listing_id")
    feat = pd.DataFrame({
        "rev_total": g["n_reviews"].sum(),
        "rev_last12": past[past["month"] >= cutoff - pd.DateOffset(months=12)].groupby("listing_id")["n_reviews"].sum(),
        "rev_last3": past[past["month"] >= cutoff - pd.DateOffset(months=3)].groupby("listing_id")["n_reviews"].sum(),
        "rev_same_season": past[(past["month"] >= cutoff - pd.DateOffset(months=12))
                                & (past["month"] < cutoff - pd.DateOffset(months=9))].groupby("listing_id")["n_reviews"].sum(),
        "months_since_first": (cutoff - g["month"].min()).dt.days / 30.44,
        "months_since_last": (cutoff - g["month"].max()).dt.days / 30.44,
    }).fillna(0)
    table = listings.set_index("id").join(feat, how="inner")
    table["target"] = future.groupby("listing_id")["n_reviews"].sum().reindex(table.index).fillna(0)
    return table


past_only = ["rev_total", "rev_last12", "rev_last3", "rev_same_season", "months_since_first",
             "months_since_last", "accommodates"]
demand = demand_table("2025-07-01")
y_d = np.log1p(demand["target"])
print(len(demand), "listings; median target", demand["target"].median(),
      "; share without reviews next year", round((demand["target"] == 0).mean(), 3))
r2_past = cross_val_score(gbm, demand[past_only], y_d, cv=cv, groups=demand["host_id"], scoring="r2").mean()
naive = np.log1p(demand["rev_last12"])
print("past-only model R²", round(r2_past, 3), "| 'next year = last year' R²", round(r2_score(y_d, naive), 3))

**Task 1.** Check one listing by hand: pick an `id`, list its rows in `reviews` around the cut-off, and recompute `rev_last12`, `rev_last3` and the target.

### Out-of-time check

Cross-validation within one cut-off still mixes listings of the same period. The real use is: train on an earlier cut-off, predict a later one.

In [ ]:
earlier = demand_table("2024-07-01")
model = HistGradientBoostingRegressor(random_state=0).fit(earlier[past_only], np.log1p(earlier["target"]))
print("trained on cut-off 2024-07-01 (", len(earlier), "listings ), tested on 2025-07-01:",
      round(r2_score(y_d, model.predict(demand[past_only])), 3))

**Task 2.** Is the out-of-time R² close to the cross-validated one? Both tables contain only listings that still exist in June 2026. Why does this survivorship make both numbers optimistic, and which listings are missing?

## 2. Snapshot columns that contain the future

In [ ]:
snapshot_cols = ["number_of_reviews_ltm", "reviews_per_month", "estimated_occupancy_l365d"]
rows = [{"features": "past-only", "R²": r2_past}]
for name, extra in [("+ snapshot review columns", snapshot_cols),
                    ("+ number_of_reviews_ltm only", ["number_of_reviews_ltm"]),
                    ("+ availability_365, minimum_nights (June 2026)", ["availability_365", "minimum_nights"])]:
    r2 = cross_val_score(gbm, demand[past_only + extra], y_d, cv=cv, groups=demand["host_id"], scoring="r2").mean()
    rows.append({"features": name, "R²": r2})
print("number_of_reviews_ltm == target for", round((demand["number_of_reviews_ltm"] == demand["target"]).mean(), 3),
      "of the listings")
pd.DataFrame(rows).round(3)

**Task 3.** For each added column, state when it was measured and which period it describes. Which of them would be available on 1 July 2025?

## 3. The revenue leak in a price model

In [ ]:
listings = pd.read_parquet(AIRBNB / "listings.parquet")
# short-stay listings with a price (minimum stay < 28 nights, as in Sessions 4-6),
# without the 26 outliers priced below EUR 10 or above EUR 1,000
bnb = listings[(listings["minimum_nights"] < 28) & listings["price"].between(10, 1000)].reset_index(drop=True)
bnb["dist_km"] = np.hypot((bnb["latitude"] - 52.5219) * 111.2,      # km to Alexanderplatz
                          (bnb["longitude"] - 13.4132) * 68.0)
print(len(listings), "listings in the snapshot,", len(bnb), "in the price-model table,",
      bnb["host_id"].nunique(), "hosts")
y = np.log(bnb["price"])
hosts = bnb["host_id"]
cols = ["accommodates", "bedrooms", "beds", "bathrooms", "dist_km", "room_type", "district",
        "review_scores_rating", "number_of_reviews", "minimum_nights", "availability_365"]
base = pd.get_dummies(bnb[cols], columns=["room_type", "district"], dtype=int)
leaky = base.assign(revenue=bnb["estimated_revenue_l365d"], occupancy=bnb["estimated_occupancy_l365d"])
for name, X in [("honest", base), ("with revenue and occupancy", leaky)]:
    print(name, round(cross_val_score(gbm, X, y, cv=cv, groups=hosts, scoring="r2").mean(), 3))

A "too good to be true" check: how well does each column alone predict the price? And which features does the leaky model rely on (permutation importance on held-out hosts, Session 10)?

In [ ]:
single = {c: cross_val_score(gbm, leaky[[c]], y, cv=cv, groups=hosts, scoring="r2").mean()
          for c in ["accommodates", "bedrooms", "dist_km", "revenue", "occupancy"]}
print({k: round(v, 3) for k, v in single.items()})

tr, te = next(GroupShuffleSplit(1, test_size=0.25, random_state=0).split(leaky, groups=hosts))
fitted = HistGradientBoostingRegressor(random_state=0).fit(leaky.iloc[tr], y.iloc[tr])
perm = permutation_importance(fitted, leaky.iloc[te], y.iloc[te], scoring="r2", n_repeats=5, random_state=0)
pd.Series(perm.importances_mean, index=leaky.columns).sort_values(ascending=False).head(6).round(3)

In our run, revenue *alone* explains less of the log price than `accommodates` (R² about 0.31 against 0.38): a single-column check does not flag it. The leak works in combination, because revenue ÷ occupancy is the price, and permutation importance on held-out hosts shows it at once: shuffling revenue or occupancy destroys the model. Check combinations, not only single columns.

In [ ]:
occupied = bnb["estimated_occupancy_l365d"] > 0
ratio = bnb.loc[occupied, "estimated_revenue_l365d"] / bnb.loc[occupied, "estimated_occupancy_l365d"]
print("listings with occupancy > 0:", round(occupied.mean(), 3),
      "| revenue / occupancy equals the price (within 1 %):",
      round((np.abs(ratio / bnb.loc[occupied, "price"] - 1) < 0.01).mean(), 3))

**Task 4.** Explain in two sentences why the revenue column is a leak. Is `number_of_reviews` a leak for the price model? (Think about what it measures and when, and whether a new listing has it.)

## 4. Text statistics of the title

In [ ]:
def title_stats(title):
    t = title.fillna("")
    return pd.DataFrame({
        "title_len": t.str.len(),
        "title_upper": t.str.count(r"[A-Z]") / t.str.len().clip(lower=1),
        "title_digits": t.str.count(r"\d"),
        "says_sqm": t.str.contains(r"m²|m2|qm|sqm", case=False).astype(int),
        "says_luxury": t.str.contains(r"luxur|premium|design", case=False).astype(int),
        "says_cozy": t.str.contains(r"cozy|cosy|gemütlich|small|little", case=False).astype(int),
    }, index=title.index)


text = title_stats(bnb["name"]).assign(n_amenities=bnb["amenities"].map(lambda s: len(json.loads(s))))
print(text.describe().loc[["mean", "50%"]].round(2).to_string())
for name, X in [("base", base), ("+ title statistics", pd.concat([base, text.drop(columns="n_amenities")], axis=1)),
                ("+ title statistics + amenity count", pd.concat([base, text], axis=1))]:
    print(name, round(cross_val_score(gbm, X, y, cv=cv, groups=hosts, scoring="r2").mean(), 3))

**Task 5.** Add one statistic of your own (for example whether the title names a district, or the word "balcony"/"Balkon"). Does it help? Is it a cause of higher prices or a marker?

## 5. Decision

For each candidate feature, write down whether it may enter (a) the demand model for 1 July 2025 and (b) a price tool for new hosts, and the moment at which its value is known: `rev_last12`, `number_of_reviews_ltm`, `reviews_per_month`, `availability_365`, `estimated_revenue_l365d`, `number_of_reviews`, `review_scores_rating`, the title statistics, `n_amenities`.